In [3]:
import asyncio
import sys
import pandas as pd
from tabulate import tabulate
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport


async def main():
    transport = StreamableHttpTransport("http://mcp_server:8000/mcp")
    async with Client(transport=transport) as client:
        await client.ping()
        print("✅ Connected to MCP server")

        while True:
            sql_query = input("\nEnter your SQL query (or type 'exit' to quit): ").strip()

            if sql_query.lower() == "exit":
                print("👋 Exiting...")
                break

            try:
                # Call the query_db tool
                result = await client.call_tool("query_db", {"sql": sql_query})

                if result.data.get("success"):
                    preview_rows = result.data.get("preview_rows", [])
                    preview_cols = result.data.get("preview_columns", [])
                    total_rows = result.data.get("total_rows", len(preview_rows))
                    total_cols = result.data.get("total_columns", len(preview_cols))

                    df = pd.DataFrame(preview_rows, columns=preview_cols)
                    df.index = df.index + 1

                    # Add "..." if rows or columns truncated
                    if total_rows > len(preview_rows):
                        df.loc["..."] = ["..."] * len(df.columns)
                    if total_cols > len(preview_cols):
                        df["..."] = "..."

                    print("\n📊 Query Result (preview):")
                    print(tabulate(df, headers="keys", tablefmt="pretty"))
                    print(f"\nShowing preview of {len(preview_rows)}x{len(preview_cols)} "
                          f"(full result: {total_rows} rows x {total_cols} cols)")
                else:
                    print("❌ Query failed:", result.data.get("error", "Unknown error"))

            except Exception as e:
                print("⚠️ Error running query:", str(e))


if __name__ == "__main__":
    try:
        asyncio.run(main())
    except RuntimeError as e:
        if "asyncio.run() cannot be called" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            loop.run_until_complete(main())
        else:
            raise

✅ Connected to MCP server



Enter your SQL query (or type 'exit' to quit):  select * from employees



📊 Query Result (preview):
+-----+--------+------------+-----------+----------------------------+------------+-----+
|     | emp_id | first_name | last_name |           email            | hire_date  | ... |
+-----+--------+------------+-----------+----------------------------+------------+-----+
|  1  |   1    |   Alice    |  Johnson  | alice.johnson@example.com  | 2018-03-12 | ... |
|  2  |   2    |    Bob     |   Smith   |   bob.smith@example.com    | 2019-07-01 | ... |
|  3  |   3    |  Charlie   |   Davis   | charlie.davis@example.com  | 2020-01-20 | ... |
|  4  |   4    |   Diana    | Martinez  | diana.martinez@example.com | 2017-11-05 | ... |
|  5  |   5    |    Evan    |   Brown   |   evan.brown@example.com   | 2021-06-15 | ... |
| ... |  ...   |    ...     |    ...    |            ...             |    ...     | ... |
+-----+--------+------------+-----------+----------------------------+------------+-----+

Showing preview of 5x5 (full result: 51 rows x 7 cols)



Enter your SQL query (or type 'exit' to quit):  exit


👋 Exiting...
